# 📊 Baseline — Tanpa Seleksi Fitur
### 3 Dataset × 3 Classifier (SVM · RF · NB)

**Struktur eksperimen:**
- Dataset    : MFCC | LPCC | COMBINED
- Classifier : LinearSVC | RandomForest | GaussianNB
- CV Method  : Leave-One-Batch-Out (cross-session)
- Total      : **9 eksperimen** (3 dataset × 3 classifier)
- Seleksi    : **Tidak ada** — semua fitur dipakai langsung

> Notebook ini menggunakan struktur data, split, dan CV yang **identik** dengan
> `PSO_3_Model.ipynb` dan `GA_3_Model.ipynb` agar hasil bisa dibandingkan langsung.

## 📁 Cell 1 — Load Data

In [ ]:
import pandas as pd
import os

# ── Sesuaikan path ke lokasi file CSV di komputermu ──
MFCC_PATH = 'D:/LISA/.retest_new/MFCC_new.csv'
LPCC_PATH  = 'D:/LISA/.retest_new/LPCC_new.csv'

mfcc_df = pd.read_csv(MFCC_PATH)
lpcc_df = pd.read_csv(LPCC_PATH)

print('MFCC shape :', mfcc_df.shape)
print('LPCC shape :', lpcc_df.shape)

assert 'filename' in mfcc_df.columns, "Kolom 'filename' tidak ditemukan di MFCC!"
assert 'filename' in lpcc_df.columns, "Kolom 'filename' tidak ditemukan di LPCC!"

print('\nContoh filename (5 pertama):')
print(mfcc_df['filename'].head().tolist())
print('\n✅ Cell 1 OK — Data loaded')


## ⚙️ Cell 2 — Import & Konfigurasi Global

In [ ]:
import numpy as np
import pandas as pd
import os
import time
from datetime import datetime
import pytz

from sklearn.metrics import f1_score
from sklearn.base import clone
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
WIB = pytz.timezone('Asia/Jakarta')

BASE_DIR = 'D:/LISA/.retest_new/Data_Output_Baseline_Scaled'
os.makedirs(BASE_DIR, exist_ok=True)

# CLASSIFIERS — identik dengan PSO & GA (LinearSVC murni, tol=1e-2)
# Scaling dilakukan di Cell 3 (Opsi A), bukan di dalam pipeline
CLASSIFIERS = {
    'SVM': LinearSVC(random_state=RANDOM_STATE, max_iter=5000, tol=1e-2),
    'RF' : RandomForestClassifier(random_state=RANDOM_STATE),
    'NB' : GaussianNB(),
}

def format_time(sec):
    m = int(sec // 60)
    s = int(sec % 60)
    return f'{m}m {s}s ({sec:.1f}s)'

def get_timestamp_wib():
    return datetime.now(WIB).strftime('%H:%M:%S')

print('Cell 2 OK — Config & Classifiers siap')
print(f'   Classifier : {list(CLASSIFIERS.keys())}')
print(f'   Output dir : {BASE_DIR}')


## 📦 Cell 3 — Persiapan Data

In [ ]:
def extract_subject_from_filename(filename):
    basename = os.path.basename(str(filename)).replace('.wav', '')
    parts = basename.split('_')
    if len(parts) < 2:
        raise ValueError(f"Format filename tidak dikenali: '{filename}'")
    return parts[0]


def prepare_features(df):
    subjects  = df['filename'].apply(extract_subject_from_filename).values
    drop_cols = [c for c in ['Unnamed: 0', 'filename', 'subject', 'datatype', 'label'] if c in df.columns]
    X = df.drop(columns=drop_cols).values.astype(np.float32)
    y = df['label'].values
    return X, y, subjects


def prepare_all_datasets(mfcc_df, lpcc_df):
    for df in [mfcc_df, lpcc_df]:
        df.dropna(inplace=True)
        df.reset_index(drop=True, inplace=True)
        df['label'] = df['label'].astype(int)

    mfcc_tr = mfcc_df[mfcc_df['datatype'] == 'train']
    mfcc_te = mfcc_df[mfcc_df['datatype'] == 'test']
    lpcc_tr = lpcc_df[lpcc_df['datatype'] == 'train']
    lpcc_te = lpcc_df[lpcc_df['datatype'] == 'test']

    X_mfcc_tr, y_tr, subjects_mfcc = prepare_features(mfcc_tr)
    X_mfcc_te, y_te, _             = prepare_features(mfcc_te)
    X_lpcc_tr, _,    subjects_lpcc = prepare_features(lpcc_tr)
    X_lpcc_te, _,    _             = prepare_features(lpcc_te)

    # Opsi A: StandardScaler di-fit SEKALI di train, transform train+test
    # Identik dengan PSO & GA agar hasil baseline bisa dibandingkan langsung
    from sklearn.preprocessing import StandardScaler
    _sc_mfcc = StandardScaler().fit(X_mfcc_tr)
    X_mfcc_tr = _sc_mfcc.transform(X_mfcc_tr).astype(np.float32)
    X_mfcc_te = _sc_mfcc.transform(X_mfcc_te).astype(np.float32)
    _sc_lpcc = StandardScaler().fit(X_lpcc_tr)
    X_lpcc_tr = _sc_lpcc.transform(X_lpcc_tr).astype(np.float32)
    X_lpcc_te = _sc_lpcc.transform(X_lpcc_te).astype(np.float32)

    X_comb_tr = np.concatenate([X_mfcc_tr, X_lpcc_tr], axis=1)
    X_comb_te = np.concatenate([X_mfcc_te, X_lpcc_te], axis=1)

    datasets = {
        'MFCC'    : (X_mfcc_tr, X_mfcc_te, y_tr, y_te, subjects_mfcc),
        'LPCC'    : (X_lpcc_tr, X_lpcc_te, y_tr, y_te, subjects_lpcc),
        'COMBINED': (X_comb_tr, X_comb_te, y_tr, y_te, subjects_mfcc),
    }

    unique_s, counts_s = np.unique(subjects_mfcc, return_counts=True)
    print('\n📊 Distribusi subjek di training set (MFCC):')
    for s, c in zip(unique_s, counts_s):
        print(f'   {s}: {c} sampel')
    print(f'   -> Leave-One-Subject-Out membuat {len(unique_s)} fold per eksperimen')

    print('\n📐 Ukuran fitur per dataset:')
    for name, (Xtr, Xte, *_) in datasets.items():
        print(f'   {name:<10}: train={Xtr.shape}, test={Xte.shape}')

    return datasets


datasets = prepare_all_datasets(mfcc_df, lpcc_df)
print('\nCell 3 OK — Dataset siap (ter-scale Opsi A)')


## 🧪 Cell 4 — Evaluasi Cross-Subject (LOSO)

In [ ]:
# ──────────────────────────────────────────────────────
# CROSS-SUBJECT EVALUATION (Leave-One-Subject-Out)
#
# Identik dengan PSO & GA — evaluasi lintas sesi (B1 dan B2)
# untuk mencegah overfitting terhadap distribusi dalam sesi yang sama.
#
# Leave-One-Subject-Out:
#   Fold 1: Train=B2, Val=B1
#   Fold 2: Train=B1, Val=B2
# ──────────────────────────────────────────────────────
def evaluate_model_loso(X, y, subjects, model):
    """
    Evaluasi model dengan Leave-One-Subject-Out CV.
    Return: (mean_f1, std_f1) dari semua fold.
    """
    scores = []
    for s in np.unique(subjects):
        val_idx = np.where(subjects == s)[0]
        tr_idx  = np.where(subjects != s)[0]
        m = clone(model)
        m.fit(X[tr_idx], y[tr_idx])
        pred = m.predict(X[val_idx])
        scores.append(f1_score(y[val_idx], pred, average='weighted'))
    return float(np.mean(scores)), float(np.std(scores))


print('✅ Cell 4 OK — Fungsi LOSO CV siap')


## 🚀 Cell 5 — Eksperimen Baseline (Main Loop)

In [ ]:
runner_start = time.time()
results  = []
total    = len(datasets) * len(CLASSIFIERS)
counter  = 0

print('=' * 70)
print('🚀 BASELINE — MULAI')
print(f'   Dataset    : {list(datasets.keys())}')
print(f'   Classifier : {list(CLASSIFIERS.keys())}')
print(f'   Total      : {total} eksperimen')
print(f'   Seleksi    : Tidak ada (semua fitur dipakai)')
print('=' * 70)

for ds_name, (X_tr, X_te, y_tr, y_te, subjects_tr) in datasets.items():
    for clf_name, clf in CLASSIFIERS.items():
        counter += 1
        exp_id  = f'{ds_name}_{clf_name}'

        # ── ETA kasar antar eksperimen ──
        if counter > 1:
            elapsed_total  = time.time() - runner_start
            avg_exp_time   = elapsed_total / (counter - 1)
            remaining_exps = total - (counter - 1)
            eta_global     = avg_exp_time * remaining_exps
            eta_str        = f'ETA global: {format_time(eta_global)}'
        else:
            eta_str = 'ETA global: menghitung...'

        print(f'\n{"=" * 70}')
        print(f'🟢 [{counter}/{total}] {exp_id}  |  {eta_str}')
        print(f'   Dataset   : {ds_name} ({X_tr.shape[1]} fitur, {X_tr.shape[0]} sampel)')
        print(f'   Classifier: {clf_name}')
        print(f'   Mulai     : {get_timestamp_wib()} WIB')
        print(f'{"=" * 70}')

        exp_start = time.time()
        n_feat    = X_tr.shape[1]

        # ── 1. LOSO CV pada training set (semua fitur) ──
        f1_cv_mean, f1_cv_std = evaluate_model_loso(X_tr, y_tr, subjects_tr, clf)

        # ── 2. Evaluasi pada test set (semua fitur) ──
        model_test = clone(clf)
        model_test.fit(X_tr, y_tr)
        pred_te = model_test.predict(X_te)
        f1_test = float(f1_score(y_te, pred_te, average='weighted'))

        exp_time = time.time() - exp_start

        print(f'\n✅ SELESAI [{counter}/{total}] {exp_id}')
        print(f'   Selesai       : {get_timestamp_wib()} WIB')
        print(f'   Durasi        : {format_time(exp_time)}')
        print(f'   F1 Test       : {f1_test:.4f}')
        print(f'   F1 CV (LOSO)  : {f1_cv_mean:.4f} ± {f1_cv_std:.4f}')
        print(f'   Gap CV<->Test : {abs(f1_cv_mean - f1_test):.4f}')
        print(f'   Fitur         : {n_feat}/{n_feat} (100.0%)')

        # ── Simpan hasil — kolom lengkap, kompatibel dengan PSO & GA ──
        results.append({
            'exp_id'          : exp_id,
            'dataset'         : ds_name,
            'classifier'      : clf_name,
            'f1_test'         : round(f1_test, 4),
            'f1_cv_mean'      : round(f1_cv_mean, 4),
            'f1_cv_std'       : round(f1_cv_std, 4),
            'f1_gap'          : round(abs(f1_cv_mean - f1_test), 4),
            'best_fitness'    : round(f1_cv_mean, 4),   # baseline: fitness = f1_cv (tanpa penalti)
            'n_feat_selected' : n_feat,                  # semua fitur terpilih
            'n_feat_total'    : n_feat,
            'feat_ratio_pct'  : 100.0,
            'duration_s'      : round(exp_time, 2),
            'method'          : 'Baseline',              # kolom tambahan untuk identifikasi metode
        })

        # Simpan hasil sementara setiap eksperimen selesai
        pd.DataFrame(results).to_csv(
            os.path.join(BASE_DIR, 'baseline_results_live.csv'), index=False
        )

# ──────────────────────────────────────────────────────
# HASIL AKHIR
# ──────────────────────────────────────────────────────
df_results = (
    pd.DataFrame(results)
    .sort_values('f1_test', ascending=False)
    .reset_index(drop=True)
)

df_results.to_csv(os.path.join(BASE_DIR, 'baseline_results_final.csv'), index=False)

total_time = time.time() - runner_start
print(f'\n{"=" * 70}')
print(f'🏁 SEMUA EKSPERIMEN SELESAI — {format_time(total_time)}')
print(f'{"=" * 70}')

header = f"{'No':>3} | {'Eksperimen':<20} | {'F1 Test':>7} | {'F1 CV':>6} | {'Std':>6} | {'Gap':>6} | {'Fitur':<18} | Durasi"
print(header)
print('-' * len(header))
for i, row in df_results.iterrows():
    feat_str = f"{int(row['n_feat_selected'])}/{int(row['n_feat_total'])} ({row['feat_ratio_pct']:.1f}%)"
    print(
        f"{i+1:>3} | {row['exp_id']:<20} | "
        f"{row['f1_test']:>7.4f} | {row['f1_cv_mean']:>6.4f} | {row['f1_cv_std']:>6.4f} | {row['f1_gap']:>6.4f} | "
        f"{feat_str:<18} | {format_time(row['duration_s'])}"
    )
print('-' * len(header))

display(df_results)


## 📊 Cell 6 — Ringkasan Per Dataset & Per Classifier

In [ ]:
# Rata-rata F1 Test per dataset
print('📈 Rata-rata F1 Test per Dataset:')
for ds, grp in df_results.groupby('dataset'):
    print(f'   {ds:<10}: {grp["f1_test"].mean():.4f} ± {grp["f1_test"].std():.4f}')

print()

# Rata-rata F1 Test per classifier
print('🤖 Rata-rata F1 Test per Classifier:')
for clf, grp in df_results.groupby('classifier'):
    print(f'   {clf:<10}: {grp["f1_test"].mean():.4f} ± {grp["f1_test"].std():.4f}')

print()

# Eksperimen terbaik
best = df_results.iloc[0]
print(f'🏆 Terbaik : {best["exp_id"]}')
print(f'   F1 Test  : {best["f1_test"]:.4f}')
print(f'   F1 CV    : {best["f1_cv_mean"]:.4f} ± {best["f1_cv_std"]:.4f}')
print(f'   Gap      : {best["f1_gap"]:.4f}')
print(f'   Fitur    : {int(best["n_feat_selected"])}/{int(best["n_feat_total"])} ({best["feat_ratio_pct"]:.1f}%)')
